# Waymax PPO on Colab

Runs the same scripts as a local machine, on a Colab GPU. Copy the `final_project` folder (without `data/`) to `MyDrive/csce642/final_project` first, and put the two WOMD shards in `MyDrive/csce642/womd/`.

This notebook was made using the Waymo Open Dataset, provided by Waymo LLC under the Waymo Dataset License Agreement for Non-Commercial Use, available at waymo.com/open/terms, and your access and use of such work are governed by the terms and conditions therein.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/csce642/final_project

In [ ]:
!pip install -q --no-deps "git+https://github.com/waymo-research/waymax.git@main#egg=waymo-waymax"
!pip install -q dm_env immutabledict mediapy dm-tree chex tabulate
import jax; print(jax.__version__, jax.devices())

In [ ]:
# optional: copy the shards straight from Google Cloud Storage instead of uploading them
# from google.colab import auth; auth.authenticate_user()
# !mkdir -p ../womd/train ../womd/val
# !gsutil cp gs://waymo_open_dataset_motion_v_1_1_0/uncompressed/tf_example/training/training_tfexample.tfrecord-00000-of-01000 ../womd/train/
# !gsutil cp gs://waymo_open_dataset_motion_v_1_1_0/uncompressed/tf_example/validation/validation_tfexample.tfrecord-00000-of-00150 ../womd/val/

In [ ]:
!python scripts/build_cache.py --tfrecords ../womd/train/training_tfexample.tfrecord-00000-of-01000 --num 400 --out data/cache/train.pkl
!python scripts/build_cache.py --tfrecords ../womd/val/validation_tfexample.tfrecord-00000-of-00150 --num 300 --out data/cache/val.pkl

In [ ]:
# main runs; each resumes from runs/<name>/checkpoint.pkl if the runtime disconnects
for seed in [0, 1, 2]:
    !python scripts/train.py --reward balanced --seed {seed} --out runs/balanced_s{seed}
!python scripts/train.py --reward safety --seed 0 --out runs/safety_s0
!python scripts/train.py --reward progress --seed 0 --out runs/progress_s0

In [ ]:
RUNS = 'runs/balanced_s0 runs/balanced_s1 runs/balanced_s2 runs/safety_s0 runs/progress_s0'
!python scripts/evaluate.py --traffic replay --out results/replay --methods expert idm_ego pure_pursuit {RUNS}
!python scripts/evaluate.py --traffic idm --out results/idm --methods expert pure_pursuit runs/balanced_s0 runs/balanced_s1 runs/balanced_s2

In [ ]:
!python scripts/make_plots.py
!python scripts/make_videos.py --method runs/balanced_s0 --eval_csv results/replay/balanced_s0.csv --out videos/balanced_s0